In [ ]:

import torch
from open_spiel.python import rl_environment
from open_spiel.python.pytorch import dqn

# Environment
env = rl_environment.Environment("pentago")
num_actions = env.action_spec()["num_actions"]
state_size = env.observation_spec()["info_state"][0]

In [32]:
agents = [
    dqn.DQN(
        player_id=i,
        state_representation_size=state_size,
        num_actions=num_actions,
        hidden_layers_sizes=[256, 256],
        replay_buffer_capacity=100_000,
        batch_size=128,
        learning_rate=1e-3,
        update_target_network_every=500,
        learn_every=10,
        discount_factor=0.99,
        epsilon_start=1.0,
        epsilon_end=0.05,
        epsilon_decay_duration=50_000,
    )
    for i in range(2)
]

# Training loop
num_episodes = 20_000

for ep in range(num_episodes):
    time_step = env.reset()

    while not time_step.last():
        current_player = time_step.observations["current_player"]
        agent_output = agents[current_player].step(time_step)
        time_step = env.step([agent_output.action])

    # Let both agents observe the terminal state
    for agent in agents:
        agent.step(time_step)

    # Logging
    if ep % 500 == 0:
        print(f"Episode {ep}/{num_episodes} | "
              f"P0 loss: {agents[0].loss} | "
              f"P1 loss: {agents[1].loss} | ")

# Save the trained agents
torch.save(agents[0]._q_network.state_dict(), "agent0.pt")
torch.save(agents[1]._q_network.state_dict(), "agent1.pt")
print("Training complete!")

Episode 0/20000 | P0 loss: None | P1 loss: None | 


/Users/colegiusto/College/ECE270/FinalProject/.venv/lib/python3.12/site-packages/open_spiel/python/pytorch/dqn.py:338: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/python_variable_indexing.cpp:353.)
  predictions = self._q_values[list(action_indices)]


Episode 500/20000 | P0 loss: 0.26947021484375 | P1 loss: 0.3233844041824341 | 
Episode 1000/20000 | P0 loss: 0.23041459918022156 | P1 loss: 0.2523628771305084 | 
Episode 1500/20000 | P0 loss: 0.193750262260437 | P1 loss: 0.20773851871490479 | 
Episode 2000/20000 | P0 loss: 0.2021852284669876 | P1 loss: 0.21594960987567902 | 
Episode 2500/20000 | P0 loss: 0.15556234121322632 | P1 loss: 0.17978106439113617 | 
Episode 3000/20000 | P0 loss: 0.1428707391023636 | P1 loss: 0.1635991930961609 | 
Episode 3500/20000 | P0 loss: 0.12339954823255539 | P1 loss: 0.1061658263206482 | 
Episode 4000/20000 | P0 loss: 0.1292276680469513 | P1 loss: 0.11398500204086304 | 
Episode 4500/20000 | P0 loss: 0.1294885277748108 | P1 loss: 0.1475267857313156 | 
Episode 5000/20000 | P0 loss: 0.11273495852947235 | P1 loss: 0.12589329481124878 | 
Episode 5500/20000 | P0 loss: 0.15163007378578186 | P1 loss: 0.07133708149194717 | 
Episode 6000/20000 | P0 loss: 0.06915271282196045 | P1 loss: 0.16293580830097198 | 
Episode

In [56]:
from open_spiel.python.algorithms import random_agent

def eval_vs_random(trained_agent, num_episodes=1000, agent_player_id=0):
    rand_agent = random_agent.RandomAgent(
        player_id=1 - agent_player_id,
        num_actions=num_actions
    )
    
    wins, losses, draws = 0, 0, 0

    for ep in range(num_episodes):
        time_step = env.reset()

        while not time_step.last():
            current_player = time_step.observations["current_player"]
            if current_player == agent_player_id:
                action = trained_agent.step(time_step, is_evaluation=True).action
            else:
                action = rand_agent.step(time_step).action
            time_step = env.step([action])

        # Check result
        returns = time_step.observations["info_state"]  # not quite right, use env
        final_returns = env._state.returns()
        if final_returns[agent_player_id] > 0:
            wins += 1
        elif final_returns[agent_player_id] < 0:
            losses += 1
        else:
            draws += 1

        if (ep + 1) % 100 == 0:
            print(f"Episode {ep+1}: W={wins} L={losses} D={draws} | "
                  f"Winrate: {wins/(ep+1):.2%}")

    print(f"\nFinal over {num_episodes} games:")
    print(f"  Wins:   {wins}  ({wins/num_episodes:.2%})")
    print(f"  Losses: {losses}  ({losses/num_episodes:.2%})")
    print(f"  Draws:  {draws}  ({draws/num_episodes:.2%})")

# Evaluate after training
eval_vs_random(agents[0], num_episodes=500, agent_player_id=0)

Episode 100: W=57 L=36 D=7 | Winrate: 57.00%
Episode 200: W=126 L=61 D=13 | Winrate: 63.00%
Episode 300: W=186 L=95 D=19 | Winrate: 62.00%
Episode 400: W=241 L=132 D=27 | Winrate: 60.25%
Episode 500: W=303 L=164 D=33 | Winrate: 60.60%

Final over 500 games:
  Wins:   303  (60.60%)
  Losses: 164  (32.80%)
  Draws:  33  (6.60%)
